In [1]:
# Libraries
import pandas as pd
import geopandas as gpd
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform # Reprojection
from rasterio import features # Rasterizing
from rasterio.enums import MergeAlg # Rasterizing while retaining attributes
np.set_printoptions(suppress = True) # Turn off scientific notation

In [3]:
# Only run initially!!

#from zipfile import ZipFile

# Unzip the specifics tracts folder 

#pop_projections_zip = './data/population_projections/ICLUS_pop_projections.zip'
#pop_density_out = './data/population_density'

#with ZipFile(pop_projections_zip, 'r') as zObject:
    # Extract downloaded 2022 census tracts data and store in data > svi folder
#    zObject.extractall(path = pop_density_out)

In [2]:
# Read in shapefile
pop = gpd.read_file('./data/population_density/ICLUS_pop_projections.shp')[['ICLUSGEOID', 'ssp2_2020', 'ssp2_2030', 'ssp2_2050', 'ssp5_2050', 'geometry']]

print(f'There are {pop.shape[0]} rows in the population projection data')
print(f'CRS: {pop.crs}')

pop.head(2)

There are 2256 rows in the population projection data
CRS: EPSG:4269


,ICLUSGEOID,ssp2_2020,ssp2_2030,ssp2_2050,ssp5_2050,geometry
0,04001,73542,76889,79450,88048,"POLYGON ((-110.00068 36.99797, -109.93414 36.9..."
1,04012,19863,19948,22000,21928,"POLYGON ((-114.73122 33.30404, -114.7299 33.30..."


In [3]:
# Reproject pop to NAD 83 CONUS Albers (epsg:5070)
pop = pop.to_crs('epsg:5070')
print(f'CRS: {pop.crs}')

CRS: epsg:5070


In [4]:
# Check for null values - none!
pop.isnull().values.any()

False

In [5]:
# Print min and max pop projection values

# ssp2 rcp45 2020
print(f"Min ssp2 rcp45 2020: {pop['ssp2_2020'].min()}")
print(f"Max ssp2 rcp45 2020: {pop['ssp2_2020'].max()}")
print('\n')

# ssp2 rcp45 2030
print(f"Min ssp2 rcp45 2030: {pop['ssp2_2030'].min()}")
print(f"Max ssp2 rcp45 2030: {pop['ssp2_2030'].max()}")
print('\n')

# ssp2 rcp45 2050
print(f"Min ssp2 rcp45 2050: {pop['ssp2_2050'].min()}")
print(f"Max ssp2 rcp45 2050: {pop['ssp2_2050'].max()}")
print('\n')

# ssp2 rcp85 2050
print(f"Min ssp5 rcp85 2050: {pop['ssp5_2050'].min()}")
print(f"Max ssp5 rcp85 2050: {pop['ssp5_2050'].max()}")

Min ssp2 rcp45 2020: 1039
Max ssp2 rcp45 2020: 21006105


Min ssp2 rcp45 2030: 1385
Max ssp2 rcp45 2030: 23311267


Min ssp2 rcp45 2050: 1959
Max ssp2 rcp45 2050: 27867591


Min ssp5 rcp85 2050: 905
Max ssp5 rcp85 2050: 35551109


In [5]:
# Average ssp2 2020 and 2030 values to calculate ssp2 2025
pop['ssp2_2025'] = (pop['ssp2_2020'] + pop['ssp2_2030']) / 2
pop.head()

,ICLUSGEOID,ssp2_2020,ssp2_2030,ssp2_2050,ssp5_2050,geometry,ssp2_2025
0,04001,73542,76889,79450,88048,"POLYGON ((-1229880.912 1641288.226, -1224073.7...",75215.5
1,04012,19863,19948,22000,21928,"POLYGON ((-1721663.073 1307456.053, -1721506.2...",19905.5
2,06003,1588,2008,2795,1356,"POLYGON ((-2051788.302 2002857.576, -2051299.0...",1798.0
3,06005,35758,34095,31197,32954,"POLYGON ((-2137147.781 2002315.662, -2137016.7...",34926.5
4,06009,43138,41388,37951,40397,"POLYGON ((-2142427.175 1971471.777, -2142300.3...",42263.0


In [7]:
# Print min and max pop projection values

# ssp2 rcp45 2025
print(f"Min ssp2 rcp45 2025: {pop['ssp2_2025'].min()}")
print(f"Max ssp2 rcp45 2025: {pop['ssp2_2025'].max()}")

Min ssp2 rcp45 2025: 1212.0
Max ssp2 rcp45 2025: 22158686.0


In [6]:
# Calculate area per spatial unit in km^2
pop['area_km_2'] = pop['geometry'].area / 10**6
pop.head()

,ICLUSGEOID,ssp2_2020,ssp2_2030,ssp2_2050,ssp5_2050,geometry,ssp2_2025,area_km_2
0,04001,73542,76889,79450,88048,"POLYGON ((-1229880.912 1641288.226, -1224073.7...",75215.5,29057.058265
1,04012,19863,19948,22000,21928,"POLYGON ((-1721663.073 1307456.053, -1721506.2...",19905.5,11688.483262
2,06003,1588,2008,2795,1356,"POLYGON ((-2051788.302 2002857.576, -2051299.0...",1798.0,1924.417464
3,06005,35758,34095,31197,32954,"POLYGON ((-2137147.781 2002315.662, -2137016.7...",34926.5,1569.513724
4,06009,43138,41388,37951,40397,"POLYGON ((-2142427.175 1971471.777, -2142300.3...",42263.0,2685.743326


In [7]:
# Calculate population density for each scenario

# ssp2 2025
pop['ssp2_2025_density'] = pop['ssp2_2025'] / pop['area_km_2']

# ssp2 2050
pop['ssp2_2050_density'] = pop['ssp2_2050'] / pop['area_km_2']

# ssp5 2050
pop['ssp5_2050_density'] = pop['ssp5_2050'] / pop['area_km_2']

pop.head()

,ICLUSGEOID,ssp2_2020,ssp2_2030,ssp2_2050,ssp5_2050,geometry,ssp2_2025,area_km_2,ssp2_2025_density,ssp2_2050_density,ssp5_2050_density
0,04001,73542,76889,79450,88048,"POLYGON ((-1229880.912 1641288.226, -1224073.7...",75215.5,29057.058265,2.588545,2.734275,3.030176
1,04012,19863,19948,22000,21928,"POLYGON ((-1721663.073 1307456.053, -1721506.2...",19905.5,11688.483262,1.703001,1.882195,1.876035
2,06003,1588,2008,2795,1356,"POLYGON ((-2051788.302 2002857.576, -2051299.0...",1798.0,1924.417464,0.934309,1.452388,0.704629
3,06005,35758,34095,31197,32954,"POLYGON ((-2137147.781 2002315.662, -2137016.7...",34926.5,1569.513724,22.253071,19.876857,20.996312
4,06009,43138,41388,37951,40397,"POLYGON ((-2142427.175 1971471.777, -2142300.3...",42263.0,2685.743326,15.736053,14.130539,15.041274


In [8]:
# Save shp
pop.to_file('./data/pop_density.shp')

/tmp/ipykernel_3423768/4046173551.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  pop.to_file('./data/pop_density.shp')
/projects/bfqp/cchan2/gsi_env/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ssp2_2025_density' to 'ssp2_2025_'
  ogr_write(
/projects/bfqp/cchan2/gsi_env/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ssp2_2050_density' to 'ssp2_2050_'
  ogr_write(
/projects/bfqp/cchan2/gsi_env/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ssp5_2050_density' to 'ssp5_2050_'
  ogr_write(


In [10]:
# Min, max pop densities ssp2 2025
print(f"Min ssp2 2025 pop density: {pop['ssp2_2025_density'].min()}")
print(f"Max ssp2 2025 pop density: {pop['ssp2_2025_density'].max()}")

# Calculate proportional change in population density for ssp2 2050 and ssp5 2050 relative to ssp2 2025

# ssp2 2050
pop['ssp2_2050_density_CHANGE'] = (pop['ssp2_2050_density'] - pop['ssp2_2025_density']) / pop['ssp2_2025_density']
print(f"Min ssp2 2050 pop density change: {pop['ssp2_2050_density_CHANGE'].min()}")
print(f"Max ssp2 2050 pop density change: {pop['ssp2_2050_density_CHANGE'].max()}")

# ssp5 2050
pop['ssp5_2050_density_CHANGE'] = (pop['ssp5_2050_density'] - pop['ssp2_2025_density']) / pop['ssp2_2025_density']
print(f"Min ssp5 2050 pop density change: {pop['ssp5_2050_density_CHANGE'].min()}")
print(f"Max ssp5 2050 pop density change: {pop['ssp5_2050_density_CHANGE'].max()}")

pop.head()

Min ssp2 2025 pop density: 0.1892630018743206
Max ssp2 2025 pop density: 1223.634908891973
Min ssp2 2050 pop density change: -0.21835783434922904
Max ssp2 2050 pop density change: 1.9183055975794254
Min ssp5 2050 pop density change: -0.3886759068121498
Max ssp5 2050 pop density change: 1.8687221069834685


,ICLUSGEOID,ssp2_2020,ssp2_2030,ssp2_2050,ssp5_2050,geometry,ssp2_2025,area_km_2,ssp2_2025_density,ssp2_2050_density,ssp5_2050_density,ssp2_2050_density_CHANGE,ssp5_2050_density_CHANGE
0,04001,73542,76889,79450,88048,"POLYGON ((-1229880.912 1641288.226, -1224073.7...",75215.5,29057.058265,2.588545,2.734275,3.030176,0.056298,0.170610
1,04012,19863,19948,22000,21928,"POLYGON ((-1721663.073 1307456.053, -1721506.2...",19905.5,11688.483262,1.703001,1.882195,1.876035,0.105222,0.101605
2,06003,1588,2008,2795,1356,"POLYGON ((-2051788.302 2002857.576, -2051299.0...",1798.0,1924.417464,0.934309,1.452388,0.704629,0.554505,-0.245829
3,06005,35758,34095,31197,32954,"POLYGON ((-2137147.781 2002315.662, -2137016.7...",34926.5,1569.513724,22.253071,19.876857,20.996312,-0.106781,-0.056476
4,06009,43138,41388,37951,40397,"POLYGON ((-2142427.175 1971471.777, -2142300.3...",42263.0,2685.743326,15.736053,14.130539,15.041274,-0.102028,-0.044152


In [ ]:
#### ssp2 2025 ###############################################################################################

In [41]:
## ssp2 2025
# Rasterize ssp2 2025 population density projection values matching dist to gwt raster 
dist_gwt_REF = './data/SSURGO_raw/dist_GWT/gwt_inches.tif'
ssp2_2025_density = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density.tif'

# Use a list, not a generator
geom_value = list(zip(pop.geometry, pop['ssp2_2025_density']))

with rasterio.open(dist_gwt_REF) as ref:
    # Copy profile
    profile = ref.profile.copy()
    
    # Update profile
    profile.update(compress = 'lzw')
        
    # Rasterize 
    rasterized = features.rasterize(
        geom_value,
        out_shape = ref.shape,
        transform = ref.transform,
        fill = -10, # Background fill
        all_touched = True,
        merge_alg = MergeAlg.replace,
        dtype = rasterio.float32
    )
    
    with rasterio.open(ssp2_2025_density, 'w', **profile) as dst:
        dst.write(rasterized, 1)

In [42]:
# Verify rasterization (WITH MASKING)

ssp2_2025_density = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density.tif'

with rasterio.open(ssp2_2025_density, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [43]:
# Remove "excess" cells 
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp2_2025_density = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density.tif'
ssp2_2025_density_MATCH = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32)

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp2_2025_density) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp2_2025_density_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [44]:
# Verify match (WITH MASKING)

ssp2_2025_density_MATCH = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density_MATCH.tif'

with rasterio.open(ssp2_2025_density_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [2]:
# Apply min-max scaling  

ssp2_2025_density_MATCH = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_raw_density_MATCH.tif'
ssp2_2025_density_standardized = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif'

with rasterio.open(ssp2_2025_density_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp2_2025_density_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.1892630010843277
Global max: 1223.6348876953125


In [3]:
# Verify min-max scaling

ssp2_2025_density_standardized = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif'

with rasterio.open(ssp2_2025_density_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [ ]:
#### ssp2 2050 ###############################################################################################

In [11]:
## ssp2 2050
# Rasterize ssp2 2050 population density CHANGE values matching dist to gwt raster 
dist_gwt_REF = './data/SSURGO_raw/dist_GWT/gwt_inches.tif'
ssp2_2050_density_change = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change.tif'

# Use a list, not a generator
geom_value = list(zip(pop.geometry, pop['ssp2_2050_density_CHANGE']))

with rasterio.open(dist_gwt_REF) as ref:
    # Copy profile
    profile = ref.profile.copy()
    
    # Update profile
    profile.update(compress = 'lzw')
        
    # Rasterize 
    rasterized = features.rasterize(
        geom_value,
        out_shape = ref.shape,
        transform = ref.transform,
        fill = -10, # Background fill
        all_touched = True,
        merge_alg = MergeAlg.replace,
        dtype = rasterio.float32
    )
    
    with rasterio.open(ssp2_2050_density_change, 'w', **profile) as dst:
        dst.write(rasterized, 1)

In [12]:
# Verify rasterization (WITH MASKING)

ssp2_2050_density_change = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change.tif'

with rasterio.open(ssp2_2050_density_change, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [13]:
# Remove "excess" cells 
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp2_2050_density_change = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change.tif'
ssp2_2050_density_change_MATCH = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32)

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp2_2050_density_change) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp2_2050_density_change_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [14]:
# Verify matching (WITH MASKING)

ssp2_2050_density_change_MATCH = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_MATCH.tif'

with rasterio.open(ssp2_2050_density_change_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [4]:
# Apply min-max scaling  

ssp2_2050_density_change_MATCH = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_MATCH.tif'
ssp2_2050_density_change_standardized = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_standardized.tif'

with rasterio.open(ssp2_2050_density_change_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp2_2050_density_change_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: -0.21835783123970032
Global max: 1.9183056354522705


In [5]:
# Verify min-max scaling

ssp2_2050_density_change_standardized = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_standardized.tif'

with rasterio.open(ssp2_2050_density_change_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [ ]:
#### ssp5 2050 ###############################################################################################

In [15]:
## ssp2 2050
# Rasterize ssp2 2050 population density CHANGE values matching dist to gwt raster 
dist_gwt_REF = './data/SSURGO_raw/dist_GWT/gwt_inches.tif'
ssp5_2050_density_change = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change.tif'

# Use a list, not a generator
geom_value = list(zip(pop.geometry, pop['ssp5_2050_density_CHANGE']))

with rasterio.open(dist_gwt_REF) as ref:
    # Copy profile
    profile = ref.profile.copy()
    
    # Update profile
    profile.update(compress = 'lzw')
        
    # Rasterize 
    rasterized = features.rasterize(
        geom_value,
        out_shape = ref.shape,
        transform = ref.transform,
        fill = -10, # Background fill
        all_touched = True,
        merge_alg = MergeAlg.replace,
        dtype = rasterio.float32
    )
    
    with rasterio.open(ssp5_2050_density_change, 'w', **profile) as dst:
        dst.write(rasterized, 1)

In [16]:
# Verify rasterization (WITH MASKING)

ssp5_2050_density_change = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change.tif'

with rasterio.open(ssp5_2050_density_change, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [17]:
# Remove "excess" cells 
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp5_2050_density_change = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change.tif'
ssp5_2050_density_change_MATCH = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32)

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp5_2050_density_change) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp5_2050_density_change_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [18]:
# Verify match (WITH MASKING)

ssp5_2050_density_change_MATCH = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_MATCH.tif'

with rasterio.open(ssp5_2050_density_change_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [6]:
# Apply min-max scaling  

ssp5_2050_density_change_MATCH = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_MATCH.tif'
ssp5_2050_density_change_standardized = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_standardized.tif'

with rasterio.open(ssp5_2050_density_change_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp5_2050_density_change_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: -0.38867589831352234
Global max: 1.8687220811843872


In [7]:
# Verify min-max scaling

ssp5_2050_density_change_standardized = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_standardized.tif'

with rasterio.open(ssp5_2050_density_change_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu